# Load Dataset

In [21]:
%run capstone_EDA.ipynb

berhasil membaca file CSV.
jumlah data dan column: (4803, 20)
list Column: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity           

,Missing Count,Missing Percentage
homepage,3091,64.355611
tagline,844,17.572351
overview,3,0.062461
runtime,2,0.041641
release_date,1,0.020820


homepage dropped (if present).
Missing values after fill:
tagline         0
overview        0
runtime         0
release_date    0
dtype: int64


In [22]:
df_capstone.head()

,budget,id,original_language,original_title,overview,popularity,release_date,revenue,runtime,status,tagline,title,vote_average,vote_count,genres,keywords,production_companies,production_countries,spoken_languages
0,237000000,19995,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,2009-12-10,2787965087,162.0,Released,Enter the World of Pandora.,Avatar,7.2,11800,"Action, Adventure, Fantasy, Science Fiction","culture clash, future, space war, space colony...","Ingenious Film Partners, Twentieth Century Fox...","United States of America, United Kingdom","English, Español"
1,300000000,285,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,2007-05-19,961000000,169.0,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,"Adventure, Fantasy, Action","ocean, drug abuse, exotic island, east india t...","Walt Disney Pictures, Jerry Bruckheimer Films,...",United States of America,English
2,245000000,206647,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,2015-10-26,880674609,148.0,Released,A Plan No One Escapes,Spectre,6.3,4466,"Action, Adventure, Crime","spy, based on novel, secret agent, sequel, mi6...","Columbia Pictures, Danjaq, B24","United Kingdom, United States of America","Français, English, Español, Italiano, Deutsch"
3,250000000,49026,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,2012-07-16,1084939099,165.0,Released,The Legend Ends,The Dark Knight Rises,7.6,9106,"Action, Crime, Drama, Thriller","dc comics, crime fighter, terrorist, secret id...","Legendary Pictures, Warner Bros., DC Entertain...",United States of America,English
4,260000000,49529,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,2012-03-07,284139100,132.0,Released,"Lost in our world, found in another.",John Carter,6.1,2124,"Action, Adventure, Science Fiction","based on novel, mars, medallion, space travel,...",Walt Disney Pictures,United States of America,English


# Metadata Fusion — combine_feature

In [23]:
# Combine overview, genres, keywords, and tagline into a single text feature for TF-IDF
try:
    feature_cols = ['overview', 'genres', 'keywords', 'tagline']
    feature_parts = []

    for col in feature_cols:
        if col in df_capstone.columns:
            feature_parts.append(df_capstone[col].fillna('').astype(str))
        else:
            feature_parts.append(pd.Series('', index=df_capstone.index))
            print(f"Warning: kolom '{col}' tidak ditemukan, menggunakan string kosong.")

    df_capstone['combine_feature'] = (
        feature_parts[0] + ' ' + feature_parts[1] + ' ' + feature_parts[2] + ' ' + feature_parts[3]
    ).str.replace(r'\s+', ' ', regex=True).str.strip()

    print('Kolom combine_feature berhasil dibuat.')
    display(df_capstone[['combine_feature']].head(3))
except NameError:
    print('Variable df_capstone not found. Jalankan sel pemuatan data terlebih dahulu.')

Kolom combine_feature berhasil dibuat.


,combine_feature
0,"In the 22nd century, a paraplegic Marine is di..."
1,"Captain Barbossa, long believed to be dead, ha..."
2,A cryptic message from Bond’s past sends him o...


# Text Preprocessing — NLP pipeline

In [24]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download resource NLTK (cukup sekali)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))  # 179 kata, lebih lengkap

def preprocess_text(text):
    if pd.isna(text):
        return ''
    text   = str(text).lower()                     # case folding
    text   = re.sub(r'[^a-z\s]', ' ', text)        # hapus tanda baca & angka
    tokens = text.split()                          # tokenisasi sederhana
    tokens = [
        lemmatizer.lemmatize(w)                    # lemmatization
        for w in tokens
        if w not in stop_words and len(w) > 2      # stopword removal
    ]
    return ' '.join(tokens)

# Terapkan ke DataFrame
df_capstone['clean_feature'] = df_capstone['combine_feature'].apply(preprocess_text)

# Verifikasi
print(f'✓ {len(df_capstone):,} baris diproses')
print(df_capstone[['combine_feature', 'clean_feature']].head(3))

✓ 4,803 baris diproses
                                     combine_feature  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   

                                       clean_feature  
0  century paraplegic marine dispatched moon pand...  
1  captain barbossa long believed dead come back ...  
2  cryptic message bond past sends trail uncover ...  


# TF-IDF Vectorization


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Parameter tuning untuk TF-IDF
# Test berbagai kombinasi parameter untuk performa optimal
param_configs = [
    {'min_df': 2, 'max_df': 0.8, 'max_features': 3000},   # Config 1: Standard
    {'min_df': 1, 'max_df': 0.9, 'max_features': 5000},   # Config 2: Include rare words
    {'min_df': 5, 'max_df': 0.7, 'max_features': 2000},   # Config 3: Conservative
    {'min_df': 2, 'max_df': 0.8, 'max_features': 4000},   # Config 4: Balanced
]

try:
    if 'clean_feature' not in df_capstone.columns:
        raise KeyError('clean_feature')

    print('Testing TF-IDF configurations...\n')
    results = []
    
    for idx, params in enumerate(param_configs):
        vectorizer = TfidfVectorizer(**params, ngram_range=(1, 2), strip_accents='unicode', lowercase=True)
        tfidf_matrix = vectorizer.fit_transform(df_capstone['clean_feature'])
        
        feature_names = vectorizer.get_feature_names_out()
        n_features = len(feature_names)
        sparsity = 1 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))
        
        results.append({
            'Config': f"Config {idx+1}",
            'min_df': params['min_df'],
            'max_df': params['max_df'],
            'max_features': params['max_features'],
            'Actual Features': n_features,
            'Matrix Shape': tfidf_matrix.shape,
            'Sparsity': f"{sparsity*100:.2f}%",
            'Memory (MB)': f"{tfidf_matrix.data.nbytes / 1024**2:.2f}"
        })
    
    results_df = pd.DataFrame(results)
    display(results_df)
    
    # Gunakan Config 1 (Standard) sebagai default yang seimbang
    print('\n✓ Menggunakan Config 1 (Standard) untuk vectorization...')
    best_params = param_configs[0]
    vectorizer_final = TfidfVectorizer(
        min_df=best_params['min_df'],
        max_df=best_params['max_df'],
        max_features=best_params['max_features'],
        ngram_range=(1, 2),
        strip_accents='unicode',
        lowercase=True
    )
    
    tfidf_matrix = vectorizer_final.fit_transform(df_capstone['clean_feature'])
    feature_names = vectorizer_final.get_feature_names_out()
    
    print(f"✓ TF-IDF Matrix berhasil dibuat")
    print(f"  - Dimensi: {tfidf_matrix.shape}")
    print(f"  - Total features: {len(feature_names)}")
    print(f"  - Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))*100:.2f}%")
    print(f"  - Non-zero elements: {tfidf_matrix.nnz:,}")
    
    # Preview top features
    print(f"\nTop 20 TF-IDF features:")
    top_features = sorted(enumerate(vectorizer_final.idf_), key=lambda x: x[1], reverse=True)[:20]
    for idx, idf_score in top_features:
        print(f"  - {feature_names[idx]}: {idf_score:.4f}")
    
except NameError:
    print('✗ Variable df_capstone tidak ditemukan. Jalankan sel pemuatan data terlebih dahulu.')
except KeyError:
    print('✗ Kolom clean_feature tidak ditemukan. Jalankan sel text preprocessing terlebih dahulu.')
except Exception as e:
    print(f'✗ Error: {e}')


Testing TF-IDF configurations...



,Config,min_df,max_df,max_features,Actual Features,Matrix Shape,Sparsity,Memory (MB)
0,Config 1,2,0.8,3000,3000,"(4803, 3000)",98.83%,1.29
1,Config 2,1,0.9,5000,5000,"(4803, 5000)",99.21%,1.44
2,Config 3,5,0.7,2000,2000,"(4803, 2000)",98.41%,1.17
3,Config 4,2,0.8,4000,4000,"(4803, 4000)",99.06%,1.38



✓ Menggunakan Config 1 (Standard) untuk vectorization...
✓ TF-IDF Matrix berhasil dibuat
  - Dimensi: (4803, 3000)
  - Total features: 3000
  - Sparsity: 98.83%
  - Non-zero elements: 169,276

Top 20 TF-IDF features:
  - joel: 7.6854
  - snow white: 7.6854
  - jedi: 7.3978
  - jigsaw: 7.3978
  - albert: 7.2800
  - frankenstein: 7.2800
  - frankie: 7.2800
  - harold: 7.2800
  - jamie: 7.2800
  - ralph: 7.2800
  - sir: 7.2800
  - wine: 7.2800
  - batman: 7.1746
  - bishop: 7.1746
  - calvin: 7.1746
  - chicken: 7.1746
  - crocodile: 7.1746
  - elvis: 7.1746
  - freddy: 7.1746
  - gotham: 7.1746


# Cosine Similarity Matrix


In [26]:
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import os
from pathlib import Path

try:
    if 'tfidf_matrix' not in locals():
        raise NameError('tfidf_matrix')
    
    print('Menghitung cosine similarity matrix...')
    print(f'Matrix dimensi: {tfidf_matrix.shape}')
    
    # Hitung cosine similarity antar semua pasangan film
    # Menggunakan scipy sparse matrix untuk efisiensi memory
    print('Processing...')
    cosine_sim_matrix = cosine_similarity(tfidf_matrix, dense_output=False)
    
    print(f'✓ Cosine similarity matrix berhasil dibuat')
    print(f'  - Dimensi: {cosine_sim_matrix.shape}')
    print(f'  - Tipe: {type(cosine_sim_matrix)}')
    print(f'  - Sparsity: {(1 - cosine_sim_matrix.nnz / (cosine_sim_matrix.shape[0] * cosine_sim_matrix.shape[1]))*100:.2f}%')
    print(f'  - Non-zero elements: {cosine_sim_matrix.nnz:,}')
    
    # Tentukan path untuk menyimpan pickle file
    base_path = Path(r"D:\File (Acer Predator 2024)\My Campus\Semester\Complete\Semester 6 ()\Proyek Sains Data (Capstone)\Tugas Kelompok\capstone_code")
    pkl_file = base_path / "cosine_similarity_matrix.pkl"
    
    # Simpan ke pickle file
    print(f'\nMenyimpan ke file: {pkl_file}')
    with open(pkl_file, 'wb') as f:
        pickle.dump(cosine_sim_matrix, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size_mb = os.path.getsize(pkl_file) / (1024**2)
    print(f'✓ File berhasil disimpan')
    print(f'  - Ukuran file: {file_size_mb:.2f} MB')
    
    # Verifikasi dengan membaca kembali dari file
    print(f'\nVerifikasi: Membaca kembali dari file...')
    with open(pkl_file, 'rb') as f:
        loaded_matrix = pickle.load(f)
    
    print(f'✓ File berhasil dibaca kembali')
    print(f'  - Dimensi: {loaded_matrix.shape}')
    print(f'  - Cocok dengan yang disimpan: {(loaded_matrix != cosine_sim_matrix).nnz == 0}')
    
    # Preview beberapa nilai similarity
    print(f'\nContoh nilai cosine similarity (film pertama dengan 5 film terdekat):')
    first_movie_sim = cosine_sim_matrix[0].toarray().flatten()
    top_indices = (-first_movie_sim).argsort()[:6]  # 6 termasuk diri sendiri
    for idx, film_idx in enumerate(top_indices):
        similarity_score = first_movie_sim[film_idx]
        film_title = df_capstone.loc[film_idx, 'title'] if 'title' in df_capstone.columns else f'Film {film_idx}'
        print(f'  {idx}. {film_title}: {similarity_score:.4f}')
    
except NameError as e:
    print(f'✗ Error: {e}. Jalankan sel TF-IDF vectorization terlebih dahulu.')
except Exception as e:
    print(f'✗ Error: {e}')


Menghitung cosine similarity matrix...
Matrix dimensi: (4803, 3000)
Processing...
✓ Cosine similarity matrix berhasil dibuat
  - Dimensi: (4803, 4803)
  - Tipe: <class 'scipy.sparse._csr.csr_matrix'>
  - Sparsity: 20.21%
  - Non-zero elements: 18,407,059

Menyimpan ke file: D:\File (Acer Predator 2024)\My Campus\Semester\Complete\Semester 6 ()\Proyek Sains Data (Capstone)\Tugas Kelompok\capstone_code\cosine_similarity_matrix.pkl
✓ File berhasil disimpan
  - Ukuran file: 210.67 MB

Verifikasi: Membaca kembali dari file...
✓ File berhasil dibaca kembali
  - Dimensi: (4803, 4803)
  - Cocok dengan yang disimpan: True

Contoh nilai cosine similarity (film pertama dengan 5 film terdekat):
  0. Avatar: 1.0000
  1. Aliens: 0.4560
  2. Alien³: 0.3940
  3. Moonraker: 0.3894
  4. Spaceballs: 0.3740
  5. Alien: 0.3515


# Bayesian Weighted Rating

In [27]:
# Bayesian weighted rating + min-max normalization
try:
    # Pastikan data tersedia
    if 'vote_average' not in df_capstone.columns or 'vote_count' not in df_capstone.columns:
        raise KeyError('vote_average atau vote_count')

    df_capstone['vote_average'] = df_capstone['vote_average'].fillna(0).astype(float)
    df_capstone['vote_count'] = df_capstone['vote_count'].fillna(0).astype(float)

    # Parameter Bayesian smoothing
    C = df_capstone['vote_average'].mean()
    m = df_capstone['vote_count'].quantile(0.75)

    df_capstone['bayes_weighted_rating'] = (
        (df_capstone['vote_count'] / (df_capstone['vote_count'] + m)) * df_capstone['vote_average'] +
        (m / (df_capstone['vote_count'] + m)) * C
    )

    # Normalisasi ke [0, 1] menggunakan min-max scaling
    min_score = df_capstone['bayes_weighted_rating'].min()
    max_score = df_capstone['bayes_weighted_rating'].max()
    if max_score > min_score:
        df_capstone['bayes_weighted_rating_norm'] = (
            df_capstone['bayes_weighted_rating'] - min_score
        ) / (max_score - min_score)
    else:
        df_capstone['bayes_weighted_rating_norm'] = 0.0

    print('✓ Bayesian weighted rating berhasil dihitung.')
    print(f'  - C (average vote_average): {C:.4f}')
    print(f'  - m (75th percentile vote_count): {m:.1f}')
    print(f'  - Normalisasi range: [{df_capstone["bayes_weighted_rating_norm"].min():.4f}, {df_capstone["bayes_weighted_rating_norm"].max():.4f}]')
    display(df_capstone[['title', 'vote_average', 'vote_count', 'bayes_weighted_rating', 'bayes_weighted_rating_norm']].sort_values('bayes_weighted_rating_norm', ascending=False).head(5))
except NameError:
    print('✗ Variable df_capstone tidak ditemukan. Jalankan sel pemuatan data terlebih dahulu.')
except KeyError as e:
    print(f'✗ Kolom tidak ditemukan: {e}. Pastikan vote_average dan vote_count ada.')
except Exception as e:
    print(f'✗ Error: {e}')


✓ Bayesian weighted rating berhasil dihitung.
  - C (average vote_average): 6.0922
  - m (75th percentile vote_count): 737.0
  - Normalisasi range: [0.0000, 1.0000]


,title,vote_average,vote_count,bayes_weighted_rating,bayes_weighted_rating_norm
1881,The Shawshank Redemption,8.5,8205.0,8.301547,1.000000
3337,The Godfather,8.4,5893.0,8.143459,0.954675
662,Fight Club,8.3,9413.0,8.139688,0.953594
3232,Pulp Fiction,8.3,8428.0,8.122458,0.948654
65,The Dark Knight,8.2,12002.0,8.078054,0.935924


# Sentiment Analysis

In [28]:
# Sentiment analysis (lexicon-based) and integration into hybrid scorer
import re

try:
    if 'clean_feature' not in df_capstone.columns:
        raise KeyError('clean_feature')

    # Small lexicon (examples). Extend as needed.
    positive_words = {
        'good','great','excellent','amazing','wonderful','best','love','loved','like','liked',
        'enjoy','enjoyed','fantastic','awesome','brilliant','masterpiece','refreshing','pleasing',
        'fascinating','charming','delightful','touching','moving','powerful','spectacular'
    }
    negative_words = {
        'bad','terrible','awful','boring','worst','poor','disappoint','disappointing','weak',
        'dull','predictable','flawed','unfortunate','uninteresting','mess','cliche','cliché','forgettable'
    }

    def sentiment_raw(text):
        if pd.isna(text) or str(text).strip() == '':
            return 0
        tokens = re.findall(r"\\b[a-z]+\\b", str(text).lower())
        pos = sum(1 for t in tokens if t in positive_words)
        neg = sum(1 for t in tokens if t in negative_words)
        return pos - neg

    df_capstone['sentiment_raw'] = df_capstone['clean_feature'].apply(sentiment_raw)

    # Normalize sentiment to [0,1]
    s_min = df_capstone['sentiment_raw'].min()
    s_max = df_capstone['sentiment_raw'].max()
    if s_max > s_min:
        df_capstone['sentiment_norm'] = (df_capstone['sentiment_raw'] - s_min) / (s_max - s_min)
    else:
        df_capstone['sentiment_norm'] = 0.5

    print('✓ Sentiment scores computed and normalized (sentiment_norm).')
    display(df_capstone[['clean_feature','sentiment_raw','sentiment_norm']].head(3))

    # Extended hybrid scorer supporting sentiment weight
    def hybrid_top_n_with_sentiment(movie_title, top_n=10, weight_similarity=0.6, weight_rating=0.3, weight_sentiment=0.1):
        # Validate
        if movie_title not in df_capstone['title'].values:
            raise ValueError(f"Judul '{movie_title}' tidak ditemukan di dataset.")
        total = weight_similarity + weight_rating + weight_sentiment
        if not abs(total - 1.0) < 1e-6:
            raise ValueError('The weights must sum to 1.0')

        idx = int(df_capstone.index[df_capstone['title'] == movie_title][0])
        sim_scores = cosine_sim_matrix[idx].toarray().flatten()
        rating_scores = df_capstone['bayes_weighted_rating_norm'].to_numpy()
        sentiment_scores = df_capstone['sentiment_norm'].to_numpy()

        hybrid = weight_similarity * sim_scores + weight_rating * rating_scores + weight_sentiment * sentiment_scores
        ranked = np.argsort(hybrid)[::-1]
        ranked = ranked[ranked != idx]
        top_idx = ranked[:top_n]

        return df_capstone.loc[top_idx, ['title','vote_average','vote_count','bayes_weighted_rating_norm','sentiment_norm']].assign(hybrid_score=hybrid[top_idx]).reset_index(drop=True)

    # Example
    movie_example = df_capstone.loc[0,'title']
    print(f"Top-N hybrid+sentiment untuk: {movie_example}")
    display(hybrid_top_n_with_sentiment(movie_example, top_n=10, weight_similarity=0.6, weight_rating=0.3, weight_sentiment=0.1))

except NameError:
    print('✗ Variable df_capstone not found. Jalankan sel pemuatan data dan preprocessing terlebih dahulu.')
except KeyError as e:
    print(f'✗ Kolom tidak ditemukan: {e}')
except Exception as e:
    print(f'✗ Error: {e}')


✓ Sentiment scores computed and normalized (sentiment_norm).


,clean_feature,sentiment_raw,sentiment_norm
0,century paraplegic marine dispatched moon pand...,0,0.5
1,captain barbossa long believed dead come back ...,0,0.5
2,cryptic message bond past sends trail uncover ...,0,0.5


Top-N hybrid+sentiment untuk: Avatar


,title,vote_average,vote_count,bayes_weighted_rating_norm,sentiment_norm,hybrid_score
0,Aliens,7.7,3220.0,0.741676,0.5,0.546081
1,Alien,7.9,4470.0,0.811512,0.5,0.504326
2,Interstellar,8.1,10867.0,0.905654,0.5,0.469673
3,2001: A Space Odyssey,7.9,2998.0,0.782600,0.5,0.459050
4,The Thing,7.8,1588.0,0.700992,0.5,0.447369
5,Star Trek,7.4,4518.0,0.688935,0.5,0.424132
6,The Empire Strikes Back,8.2,5879.0,0.903566,0.5,0.419819
7,Spaceballs,6.7,902.0,0.462467,0.5,0.413148
8,Gattaca,7.5,1808.0,0.653306,0.5,0.410804
9,Gravity,7.3,5751.0,0.673515,0.5,0.407940


# Hybrid Scoring & Top-N Ranking

In [29]:
import numpy as np

try:
    if 'cosine_sim_matrix' not in locals() and 'cosine_sim_matrix' not in globals():
        raise NameError('cosine_sim_matrix')
    if 'bayes_weighted_rating_norm' not in df_capstone.columns:
        raise KeyError('bayes_weighted_rating_norm')
    if 'title' not in df_capstone.columns:
        raise KeyError('title')

    def hybrid_top_n(movie_title, top_n=10, weight_similarity=0.7, weight_rating=0.3):
        if movie_title not in df_capstone['title'].values:
            raise ValueError(f"Judul '{movie_title}' tidak ditemukan di dataset.")

        if not np.isclose(weight_similarity + weight_rating, 1.0):
            raise ValueError('weight_similarity + weight_rating harus sama dengan 1.0')

        idx = int(df_capstone.index[df_capstone['title'] == movie_title][0])
        similarity_scores = cosine_sim_matrix[idx].toarray().flatten()

        rating_scores = df_capstone['bayes_weighted_rating_norm'].to_numpy()
        hybrid_scores = (weight_similarity * similarity_scores) + (weight_rating * rating_scores)

        ranked_indices = np.argsort(hybrid_scores)[::-1]
        ranked_indices = ranked_indices[ranked_indices != idx]
        top_indices = ranked_indices[:top_n]

        return df_capstone.loc[top_indices, ['title', 'vote_average', 'vote_count', 'bayes_weighted_rating_norm']].assign(
            hybrid_score=hybrid_scores[top_indices]
        ).reset_index(drop=True)

    # Contoh pemanggilan rekomendasi
    movie_example = df_capstone.loc[0, 'title']
    print(f"Top-N rekomendasi untuk: {movie_example}")
    display(hybrid_top_n(movie_example, top_n=10, weight_similarity=0.7, weight_rating=0.3))

except NameError as e:
    print(f'✗ Variable tidak ditemukan: {e}. Pastikan cosine similarity dan rating normalisasi sudah dihitung.')
except KeyError as e:
    print(f'✗ Kolom tidak ditemukan: {e}')
except Exception as e:
    print(f'✗ Error: {e}')


Top-N rekomendasi untuk: Avatar


,title,vote_average,vote_count,bayes_weighted_rating_norm,hybrid_score
0,Aliens,7.7,3220.0,0.741676,0.541677
1,Alien,7.9,4470.0,0.811512,0.489472
2,Interstellar,8.1,10867.0,0.905654,0.444336
3,2001: A Space Odyssey,7.9,2998.0,0.782600,0.438095
4,The Thing,7.8,1588.0,0.700992,0.428548
5,Star Trek,7.4,4518.0,0.688935,0.402041
6,Spaceballs,6.7,902.0,0.462467,0.400549
7,Alien³,6.2,1633.0,0.387862,0.392169
8,Gattaca,7.5,1808.0,0.653306,0.388273
9,The Empire Strikes Back,8.2,5879.0,0.903566,0.386277


# Simpan Model

In [34]:
import pickle

# =========================
# Simpan dataframe utama
# =========================

df_capstone.to_csv(
    "cleaned_data.csv",
    index=False
)

# =========================
# Simpan cosine similarity
# =========================

with open("cosine_similarity.pkl", "wb") as f:
    pickle.dump(cosine_sim_matrix, f)

print("data dan model berhasil disimpan")

data dan model berhasil disimpan


# Evaluasi — Precision K, Recall K, F1

In [32]:
# Evaluation (sentiment-aware): Precision@K, Recall@K, F1 (genre-based ground truth)
import random
import numpy as np

# Hybrid scoring accessor that supports sentiment (falls back if sentiment missing)
def get_hybrid_scores(idx, weight_similarity=0.6, weight_rating=0.3, weight_sentiment=0.1):
    if abs(weight_similarity + weight_rating + weight_sentiment - 1.0) > 1e-6:
        raise ValueError("Weights must sum to 1.0")

    n = len(df_capstone)
    sim = cosine_sim_matrix[idx].toarray().flatten()
    rating = df_capstone['bayes_weighted_rating_norm'].to_numpy() if 'bayes_weighted_rating_norm' in df_capstone.columns else np.zeros(n)
    sentiment = df_capstone['sentiment_norm'].to_numpy() if 'sentiment_norm' in df_capstone.columns else np.zeros(n)
    return weight_similarity * sim + weight_rating * rating + weight_sentiment * sentiment


def evaluate_precision_recall_f1_sentiment(Ks=[5,10,20], weight_similarity=0.6, weight_rating=0.3, weight_sentiment=0.1, sample_size=300, random_state=42):
    # Basic checks
    if 'genres' not in df_capstone.columns:
        raise KeyError('genres')
    if 'title' not in df_capstone.columns:
        raise KeyError('title')

    n = len(df_capstone)
    indices = list(range(n))
    random.seed(random_state)
    if sample_size is None or sample_size > n:
        sample = indices
    else:
        sample = random.sample(indices, sample_size)

    metrics = {k: {'precisions': [], 'recalls': [], 'f1s': []} for k in Ks}

    def parse_genres(val):
        if pd.isna(val) or str(val).strip() == '':
            return set()
        return set([g.strip().lower() for g in str(val).split(',') if g.strip()])

    genres_sets = [parse_genres(g) for g in df_capstone['genres']]

    for idx in sample:
        q_genres = genres_sets[idx]
        if not q_genres:
            continue
        relevant = {i for i, gs in enumerate(genres_sets) if (gs & q_genres)}
        relevant.discard(idx)
        if len(relevant) == 0:
            continue

        scores = get_hybrid_scores(idx, weight_similarity, weight_rating, weight_sentiment)
        ranked = np.argsort(scores)[::-1]
        ranked = ranked[ranked != idx]

        for k in Ks:
            topk = ranked[:k]
            hits = len(set(topk) & relevant)
            prec = hits / k
            rec = hits / len(relevant)
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
            metrics[k]['precisions'].append(prec)
            metrics[k]['recalls'].append(rec)
            metrics[k]['f1s'].append(f1)

    rows = []
    for k in Ks:
        vals = metrics[k]
        if len(vals['precisions']) == 0:
            rows.append({'K': k, 'Precision@K': None, 'Recall@K': None, 'F1@K': None, 'QueriesEvaluated': 0})
        else:
            rows.append({
                'K': k,
                'Precision@K': float(np.mean(vals['precisions'])),
                'Recall@K': float(np.mean(vals['recalls'])),
                'F1@K': float(np.mean(vals['f1s'])),
                'QueriesEvaluated': len(vals['precisions'])
            })
    results_df = pd.DataFrame(rows)
    display(results_df)
    return results_df

# Run evaluation (adjust sample_size or weights as needed)
print('Running sentiment-aware evaluation (sample_size=300)...')
results_sentiment_eval = evaluate_precision_recall_f1_sentiment(Ks=[5,10,20], weight_similarity=0.6, weight_rating=0.3, weight_sentiment=0.1, sample_size=300)
print('Evaluation completed.')


Running sentiment-aware evaluation (sample_size=300)...


,K,Precision@K,Recall@K,F1@K,QueriesEvaluated
0,5,0.728667,0.001836,0.003647,300
1,10,0.699000,0.003374,0.006658,300
2,20,0.665500,0.006151,0.012013,300


Evaluation completed.
